Cell 1: Imports & Configuration

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Define the path to your dataset
# Adjust this path if your notebook is in a different directory relative to the data
DATA_DIR = 'data/iam_dataset'

print("Libraries loaded and configuration set.")

Libraries loaded and configuration set.


Cell 2: Data Loading & Sanity Checks

In [11]:
def load_and_check_data(data_dir):
    files = {
        'identities': 'identities.parquet',
        'applications': 'applications.parquet',
        'entitlements': 'entitlements.parquet',
        'account_entitlements': 'account_entitlements.parquet'
    }

    dfs = {}

    for name, filename in files.items():
        path = os.path.join(data_dir, filename)
        if os.path.exists(path):
            print(f"Loading {name} from {path}...")
            try:
                # Explicitly specifying engine='fastparquet' to avoid pyarrow conflict
                df = pd.read_parquet(path, engine='fastparquet')
                dfs[f'df_{name}'] = df

                # Sanity Checks
                print(f"--- {name.upper()} ---")
                print(f"Shape: {df.shape}")
                # Check for missing values
                missing = df.isna().sum()
                print(f"Missing Values:\n{missing[missing > 0]}")
                print("-" * 30)
            except Exception as e:
                print(f"Error reading {filename}: {e}")
        else:
            print(f"Error: File not found at {path}")

    return dfs

# Load data
data_frames = load_and_check_data(DATA_DIR)

# Unpack dictionary to individual variables for easier access
df_identities = data_frames.get('df_identities')
df_apps = data_frames.get('df_applications')
df_entitlements = data_frames.get('df_entitlements')
df_acc_ent = data_frames.get('df_account_entitlements')

Loading identities from data/iam_dataset\identities.parquet...
--- IDENTITIES ---
Shape: (50000, 8)
Missing Values:
manager_id    5036
dtype: int64
------------------------------
Loading applications from data/iam_dataset\applications.parquet...
--- APPLICATIONS ---
Shape: (1000, 4)
Missing Values:
Series([], dtype: int64)
------------------------------
Loading entitlements from data/iam_dataset\entitlements.parquet...
--- ENTITLEMENTS ---
Shape: (10000, 5)
Missing Values:
Series([], dtype: int64)
------------------------------
Loading account_entitlements from data/iam_dataset\account_entitlements.parquet...
--- ACCOUNT_ENTITLEMENTS ---
Shape: (618015, 4)
Missing Values:
Series([], dtype: int64)
------------------------------


Cell 3: Basic Statistics

In [12]:
if df_identities is not None and df_entitlements is not None and df_acc_ent is not None:
    print("### Basic Statistics ###\n")

    # 1. Unique Counts
    # Adjust column names 'department' and 'title' if they differ in your actual schema
    if 'department' in df_identities.columns:
        print(f"Unique Departments: {df_identities['department'].nunique()}")
    if 'title' in df_identities.columns:
        print(f"Unique Job Titles: {df_identities['title'].nunique()}")

    print(f"Unique Entitlements: {df_entitlements.shape[0]}") # Assuming one row per entitlement

    # 2. Average Entitlements per User
    # We group by identity_id (or account_id) in the account_entitlements dataframe
    # Assuming 'identity_id' is the common key. If it's 'account_id', swap it here.
    group_col = 'identity_id' if 'identity_id' in df_acc_ent.columns else 'account_id'

    entitlements_per_user = df_acc_ent.groupby(group_col).size()
    avg_entitlements = entitlements_per_user.mean()

    print(f"\nAverage Entitlements per User: {avg_entitlements:.2f}")
    print(f"Max Entitlements for a single User: {entitlements_per_user.max()}")
else:
    print("Dataframes not loaded correctly. Check paths.")

### Basic Statistics ###

Unique Departments: 10
Unique Entitlements: 10000

Average Entitlements per User: 1.00
Max Entitlements for a single User: 2


Cell 4: Visualizations

In [1]:
# 1. Distribution of Identities across Departments
if df_identities is not None and 'department' in df_identities.columns:
    plt.figure(figsize=(14, 6))
    # Get top 20 departments to avoid overcrowding if there are many
    top_depts = df_identities['department'].value_counts().nlargest(20).index
    sns.countplot(data=df_identities[df_identities['department'].isin(top_depts)],
                  x='department',
                  order=top_depts,
                  palette='viridis')
    plt.title('Distribution of Identities by Department (Top 20)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# 2. Distribution of Entitlements by Risk Level
if df_entitlements is not None and 'risk_level' in df_entitlements.columns:
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df_entitlements, x='risk_level', palette='magma')
    plt.title('Distribution of Entitlements by Risk Level')
    plt.show()

# 3. Top 10 Most Assigned Entitlements
if df_acc_ent is not None:
    plt.figure(figsize=(12, 6))
    # Assuming 'entitlement_id' is the column in account_entitlements
    top_10_entitlements = df_acc_ent['entitlement_id'].value_counts().head(10)

    # Optional: Merge with df_entitlements to get friendly names if available
    # top_10_names = top_10_entitlements.rename(index=df_entitlements.set_index('id')['name'])

    sns.barplot(x=top_10_entitlements.index, y=top_10_entitlements.values, palette='coolwarm')
    plt.title('Top 10 Most Assigned Entitlements')
    plt.xlabel('Entitlement ID')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()

# 4. Histogram of Number of Entitlements per User
if df_acc_ent is not None:
    plt.figure(figsize=(10, 6))
    # Re-using the groupby from the stats cell
    group_col = 'identity_id' if 'identity_id' in df_acc_ent.columns else 'account_id'
    user_entitlement_counts = df_acc_ent.groupby(group_col).size()

    sns.histplot(user_entitlement_counts, bins=30, kde=True, color='teal')
    plt.title('Distribution of Number of Entitlements per User')
    plt.xlabel('Number of Entitlements')
    plt.ylabel('Frequency (Users)')
    plt.show()

NameError: name 'df_identities' is not defined